# Logging Training Metrics

[Resource](https://www.tensorflow.org/tensorboard/scalars_and_keras)

Before we can even think about making our own model, there's still a few prerequisites we have left. A very essential one is logging training metrics the RIGHT WAY.

After this, we will implement what we've learned for the machine learning applications on the datasets we've used thus far.

# Overview

Machine learning invariably involves understanding key metrics such as loss and how they change as training progresses. These metrics can help you understand if you're overfitting, for example, or if you're unnecessarily training for too long. You may want to compare these metrics across different training runs to help debug and improve your model.

TensorBoard's **Time Series Dashboard** allows you to visualize these metrics using a simple API with very little effort. This tutorial presents very basic examples to help you learn how to use these APIs with TensorBoard when developing your Keras model. You will learn how to use the Keras TensorBoard callback and TensorFlow Summary APIs to visualize default and custom scalars.

## Setup

In [8]:
# Load the TensorBoard notebook extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [9]:
from datetime import datetime
from packaging import version

import tensorflow as tf
from tensorflow import keras
from keras import backend as K

import numpy as np

print(tf.__version__)

2.20.0


## Clear any logs from previous runs

Run this in your terminal:

`rm -rf ./logs`

## Set up data for a simple regression

You're now going to use Keras to calculate a regression. 

**Note:** While using neural networks and gradient descent is overkill for this kind of problem, it does make for a very easy to understand example.

(Hmmm wonder if they mean that in the context of linear regression in general or just for this specific problem they're about to present?)

Note: After quickly consulting my AI associate, it seems like they're talking about this specific problem, not linear regression in general.

You're going to use TensorBoard to observe how training and test loss change across epochs. Hopefully, you'll see training and test loss decrease over time and then remain steady.

First, generate 1000 data points roughly along the line *y = 0.5x _2*. Split these data points into training and test sets. Your hope is that the neural network learns this relationship (duh).

In [10]:
data_size = 1000
train_pct = 0.8

train_size = int(data_size * train_pct)

# Create some input data between -1 and 1 and randomize it
x = np.linspace(-1, 1, data_size)
np.random.shuffle(x)

# Generate the output data
y = 0.5*x + 2 + np.random.normal(0, 0.05, (data_size, ))

# Split into test and train pairs
x_train, y_train = x[:train_size], y[:train_size]
x_test, y_test = x[train_size:], y[train_size:]

## Training the model and logging loss

You're now ready to define, train, and evaluate your model.

To log the *loss* scalar as you train, you'll do the following:
1. Create the Keras TensorBoard callback.
1. Specify a log directory.
1. Pass the TensorBoard callback to Keras' `Model.fit()`

TensorBoard reads log data from the log directory hierarchy. In this notebook, the root log directory is `logs/scalars`, suffixed by a timestamped subdirectory. The timestamped subdirectory enables you to easily identify and select training runs as you use TensorBoard and iterate on your model.

In [11]:
logdir = "logs/scalars/" + datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = keras.callbacks.TensorBoard(log_dir=logdir)

model = keras.models.Sequential([
    keras.layers.Dense(15, input_dim=1),
    keras.layers.Dense(1),
])

model.compile(
    loss="mse", # keras.losses.mean_squared_error
    optimizer=keras.optimizers.SGD(learning_rate=0.2),
)

training_history = model.fit(
    x_train,
    y_train,
    batch_size=train_size,
    verbose=0, # Suppress "chatty" output. Tensorboard takes care of this for us
    epochs=100,
    validation_data=(x_test, y_test),
    callbacks=[tensorboard_callback]
)

print("Average test loss: ", np.average(training_history.history["loss"]))

/Users/christiancamp/Desktop/Learning Machine Learning/An Introduction to Statistical Learning/ml-learning-linear-regression/lin-reg-env/lib/python3.13/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Average test loss:  0.045226812828332186


# Examining Loss Using TensorBoard

Now, start TensorBoard, specifying the root log directory you used above.

In [12]:
%tensorboard --logdir logs/scalars

Reusing TensorBoard on port 6006 (pid 89238), started 0:04:18 ago. (Use '!kill 89238' to kill it.)

Cool stuff. Love it!

You may see TensorBoard display the message "No dashboards are active for the current data set". That's because initial logging data hasn't been saved yet. As training progresses, the Keras model will start logging data. TensorBoard will periodically refresh and show you your scalar metrics. If you're impatient (lol), you can tap the Refresh arrow at the top right.

(Why is the author of this tutorial so sassy? Hahahaha)

As you watch the training progress, note how both training and validation loss rapidly decrease, and then remain stable. In fact, you could have stopped training after 25 epochs.

Hover over the graph to see specific data points. You can also try zooming in with your mouse, or selecting part of them to view more detail.

Notice the "Runs" selector on the left. a "run" represents a set of logs from a round of training, in this case the result of `Model.fit()`. Developers typically have many, many runs, as they experiment and develop their model over time.

Use the Runs selector to choose specific runs, or choose from only training or validation. Comparing runs will help you evaluate which version of your code is solving your problem better.

TensorBoard's loss graph demonstrates that the loss consistently decreased for both training and validation and then stabilized. That means that the model's metrics are likely very good! Now let's see how the model actually behaves in real life.

Given the input data (60, 25, 2), the line *y = 0.5x + 2* should yield (32, 14.5, 3). Does the model agree?

In [14]:
print(model.predict(np.array([60, 25, 2])))
# True values to compare predictions against: 
# [[32.0]
#  [14.5]
#  [ 3.0]]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step
[[32.05189  ]
 [14.520256 ]
 [ 2.9994621]]


Not bad... not bad at all.

# Logging Custom Scalars

What if you want to log custom values, such as a dynamic learning rate? To do that, you need to use the TensorFlow Summary API.

Retrain the regression model and log a custom learning rate. Here's how:
1. Create a file writer, using `tf.summary.create_file_writer()`.
1. Define a custom learning rate function. This will be passed to the Keras `LearningRateScheduler` callback.
1. Inside the learning rate function, use `tf.summary.scalar()` to log the custom learning rate.
1. Pass the LearningRateScheduler callback to `Model.fit()`.

In general, to log a custom scalar, you need to use `tf.summary.scalar()` with a file writer. The file writer is responsible for writing data for this run to the specified directory and is implicitly used when you use the `tf.sumary.scalar()`.